# Your Title Here

**Name(s)**: Baixi Guo, Chiu-Chiu (JoJo) Lin, Ryan Liao

**Website Link**: https://jol145099.github.io/Seasoned-Analysts-Recipes-and-Ratings/

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
pd.options.plotting.backend = 'plotly'

from dsc259r_utils import * # Feel free to uncomment and use this.

## Step 1: Introduction

### Source of data

PDF File --> 

<a href="/Users/steveg/Desktop/dsc259r-course-materials/projects/proj04/Final Project Datasets.pdf">RELATIVE LINK</a>

ACTUAL LINK: https://canvas.ucsd.edu/courses/71325/files/17272917

### Motivation of choosing data

We chose the food.com dataset because it pairs structured recipe metadata (cooking time, ingredient counts, nutritional info, tags) with user-generated ratings, giving us both a rich set of predictive features and a clear regression target in calories. The dataset is large enough to support meaningful permutation tests and model training. All of us also had mutual interest in the topic as well.

### Load data

In [2]:
# Define path
interaction_data_path = './RAW_interactions.csv'
recipe_data_path = './RAW_recipes.csv'

# Load CSV data
interactions_df = pd.read_csv(interaction_data_path)
recipes_df = pd.read_csv(recipe_data_path)

### Understand data

#### Semantic meaning of the columns

#### **Recipes Dataset**


| Column | Description |
| :--- | :--- |
| **`name`** | Recipe name |
| **`id`** | Recipe ID |
| **`minutes`** | Minutes to prepare recipe |
| **`contributor_id`** | User ID who submitted this recipe |
| **`submitted`** | Date recipe was submitted |
| **`tags`** | Food.com tags for recipe |
| **`nutrition`** | Nutrition information in the form [calories (#), total fat (PDV), sugar (PDV), sodium (PDV), protein (PDV), saturated fat (PDV), carbohydrates (PDV)]; PDV stands for “percentage of daily value” |
| **`n_steps`** | Number of steps in recipe |
| **`steps`** | Text for recipe steps, in order |
| **`description`** | User-provided description |

#### **Ratings Dataset**


| Column | Description |
| :--- | :--- |
| **`user_id`** | User ID |
| **`recipe_id`** | Recipe ID |
| **`date`** | Date of interaction |
| **`rating_original`** | Rating given |
| **`review`** | Review text |

In [3]:
interactions_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 731927 entries, 0 to 731926
Data columns (total 5 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   user_id    731927 non-null  int64 
 1   recipe_id  731927 non-null  int64 
 2   date       731927 non-null  object
 3   rating     731927 non-null  int64 
 4   review     731758 non-null  object
dtypes: int64(3), object(2)
memory usage: 27.9+ MB


In [4]:
recipes_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 83782 entries, 0 to 83781
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   name            83781 non-null  object
 1   id              83782 non-null  int64 
 2   minutes         83782 non-null  int64 
 3   contributor_id  83782 non-null  int64 
 4   submitted       83782 non-null  object
 5   tags            83782 non-null  object
 6   nutrition       83782 non-null  object
 7   n_steps         83782 non-null  int64 
 8   steps           83782 non-null  object
 9   description     83712 non-null  object
 10  ingredients     83782 non-null  object
 11  n_ingredients   83782 non-null  int64 
dtypes: int64(5), object(7)
memory usage: 7.7+ MB


### Brainstorm 

• What types of recipes tend to have the most calories?
• What types of recipes tend to have higher average ratings?
• What types of recipes tend to be healthier (i.e. more protein, fewer carbs)?
• What is the relationship between the cooking time and average rating of recipes?

## Step 2: Data Cleaning and Exploratory Data Analysis

### Merge datasets based on the requirement

In [5]:
# Perform recipe data left joins interaction data
merged_df = recipes_df.merge(
  interactions_df, 
  how='left', 
  left_on='id', 
  right_on='recipe_id'
)

## Specific requirement by Professor

# Replace zero rating to nan
merged_df.loc[merged_df['rating'] == 0, 'rating'] = np.nan

# Find average rating per recipe
avg_rating_per_recipe = (
    merged_df
    .groupby('id')['rating']
    .mean()
)

# Merged the info again to the merged_df
merged_df = merged_df.merge(
  avg_rating_per_recipe, 
  how='inner',
  left_on='id',
  right_index=True,
  suffixes=('', '_average')
)

In [6]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 234429 entries, 0 to 234428
Data columns (total 18 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   name            234428 non-null  object 
 1   id              234429 non-null  int64  
 2   minutes         234429 non-null  int64  
 3   contributor_id  234429 non-null  int64  
 4   submitted       234429 non-null  object 
 5   tags            234429 non-null  object 
 6   nutrition       234429 non-null  object 
 7   n_steps         234429 non-null  int64  
 8   steps           234429 non-null  object 
 9   description     234315 non-null  object 
 10  ingredients     234429 non-null  object 
 11  n_ingredients   234429 non-null  int64  
 12  user_id         234428 non-null  float64
 13  recipe_id       234428 non-null  float64
 14  date            234428 non-null  object 
 15  rating          219393 non-null  float64
 16  review          234371 non-null  object 
 17  rating_ave

In [7]:
merged_df[['submitted', 'tags', 'nutrition', 'steps', 'description', 'ingredients', 'date', 'review']].head()

,submitted,tags,nutrition,steps,description,ingredients,date,review
0,2008-10-27,"['60-minutes-or-less', 'time-to-make', 'course...","[138.4, 10.0, 50.0, 3.0, 3.0, 19.0, 6.0]",['heat the oven to 350f and arrange the rack i...,"these are the most; chocolatey, moist, rich, d...","['bittersweet chocolate', 'unsalted butter', '...",2008-11-19,"These were pretty good, but took forever to ba..."
1,2011-04-11,"['60-minutes-or-less', 'time-to-make', 'cuisin...","[595.1, 46.0, 211.0, 22.0, 13.0, 51.0, 26.0]","['pre-heat oven the 350 degrees f', 'in a mixi...",this is the recipe that we use at my school ca...,"['white sugar', 'brown sugar', 'salt', 'margar...",2012-01-26,Originally I was gonna cut the recipe in half ...
2,2008-05-30,"['60-minutes-or-less', 'time-to-make', 'course...","[194.8, 20.0, 6.0, 32.0, 22.0, 36.0, 3.0]","['preheat oven to 350 degrees', 'spray a 2 qua...",since there are already 411 recipes for brocco...,"['frozen broccoli cuts', 'cream of chicken sou...",2008-12-31,This was one of the best broccoli casseroles t...
3,2008-05-30,"['60-minutes-or-less', 'time-to-make', 'course...","[194.8, 20.0, 6.0, 32.0, 22.0, 36.0, 3.0]","['preheat oven to 350 degrees', 'spray a 2 qua...",since there are already 411 recipes for brocco...,"['frozen broccoli cuts', 'cream of chicken sou...",2009-04-13,I made this for my son's first birthday party ...
4,2008-05-30,"['60-minutes-or-less', 'time-to-make', 'course...","[194.8, 20.0, 6.0, 32.0, 22.0, 36.0, 3.0]","['preheat oven to 350 degrees', 'spray a 2 qua...",since there are already 411 recipes for brocco...,"['frozen broccoli cuts', 'cream of chicken sou...",2013-08-02,Loved this. Be sure to completely thaw the br...


### Convert string to list type

In [8]:
def str_to_list(df, col, func):
    return (
        df[col]
        # Remove spaces around the whole string just in case
        .str.strip()   
        # Remove [] around the whole string
        .str.replace('[', '')
        .str.replace(']', '')
        # Split to Python list
        .str.split(', ')
        # Apply custom lambda function
        .apply(func)
    )

In [9]:
merged_df['tag'] = merged_df.pipe(
    str_to_list, 
    'tags', 
    # For each element, strip the ''
    lambda ser: [lst_ele.strip("'") for lst_ele in ser]
)

display(merged_df['tag'])
display(type(merged_df['tag'].loc[0]))
display(type(merged_df['tag'].loc[0][0]))


0         [60-minutes-or-less, time-to-make, course, mai...
1         [60-minutes-or-less, time-to-make, cuisine, pr...
2         [60-minutes-or-less, time-to-make, course, mai...
                                ...                        
234426    [30-minutes-or-less, time-to-make, course, pre...
234427    [30-minutes-or-less, time-to-make, course, pre...
234428    [30-minutes-or-less, time-to-make, course, pre...
Name: tag, Length: 234429, dtype: object

list

str

In [10]:
merged_df['nutrition'] = merged_df.pipe(
    str_to_list, 
    'nutrition', 
    # For each element, convert them to float
    lambda ser:[float(lst_ele) for lst_ele in ser]
)

display(merged_df['nutrition'])
display(type(merged_df['nutrition'].loc[0]))
display(type(merged_df['nutrition'].loc[0][0]))

0             [138.4, 10.0, 50.0, 3.0, 3.0, 19.0, 6.0]
1         [595.1, 46.0, 211.0, 22.0, 13.0, 51.0, 26.0]
2            [194.8, 20.0, 6.0, 32.0, 22.0, 36.0, 3.0]
                              ...                     
234426        [174.9, 14.0, 33.0, 4.0, 4.0, 11.0, 6.0]
234427        [174.9, 14.0, 33.0, 4.0, 4.0, 11.0, 6.0]
234428        [174.9, 14.0, 33.0, 4.0, 4.0, 11.0, 6.0]
Name: nutrition, Length: 234429, dtype: object

list

float

In [11]:
merged_df['steps'] = merged_df.pipe(
    str_to_list, 
    'steps', 
    # For each element, strip the ''
    lambda ser: [lst_ele.strip("'") for lst_ele in ser]
)

display(merged_df['steps'])
display(type(merged_df['steps'].loc[0]))
display(type(merged_df['steps'].loc[0][0]))

0         [heat the oven to 350f and arrange the rack in...
1         [pre-heat oven the 350 degrees f, in a mixing ...
2         [preheat oven to 350 degrees, spray a 2 quart ...
                                ...                        
234426    [whip sugar and shortening in a large bowl , a...
234427    [whip sugar and shortening in a large bowl , a...
234428    [whip sugar and shortening in a large bowl , a...
Name: steps, Length: 234429, dtype: object

list

str

In [12]:
merged_df['ingredients'] = merged_df.pipe(
    str_to_list, 
    'ingredients', 
    # For each element, strip the ''
    lambda ser: [lst_ele.strip("'") for lst_ele in ser]
)

display(merged_df['ingredients'])
display(type(merged_df['ingredients'].loc[0]))
display(type(merged_df['ingredients'].loc[0][0]))

0         [bittersweet chocolate, unsalted butter, eggs,...
1         [white sugar, brown sugar, salt, margarine, eg...
2         [frozen broccoli cuts, cream of chicken soup, ...
                                ...                        
234426    [granulated sugar, shortening, eggs, flour, cr...
234427    [granulated sugar, shortening, eggs, flour, cr...
234428    [granulated sugar, shortening, eggs, flour, cr...
Name: ingredients, Length: 234429, dtype: object

list

str

### Convert to datetime type

### Flatten nested structure

In [13]:
nutr_cols = [
    "calories", "total_fat", "sugar",
    "sodium", "protein", "sat_fat", "carbs"
]
nutr_cols = ["nutrition_"+nutr for nutr in nutr_cols]

# Extract them as list of many rows (list)
nutrition_info = merged_df['nutrition'].tolist()

# Reconstruct a dataframe
# Feed it back to merged_df
# While it gives them a new columns names
merged_df[nutr_cols] = pd.DataFrame(
    data=nutrition_info,
    index=merged_df.index
)
del nutrition_info

merged_df[nutr_cols]

,nutrition_calories,nutrition_total_fat,nutrition_sugar,nutrition_sodium,nutrition_protein,nutrition_sat_fat,nutrition_carbs
0,138.4,10.0,50.0,3.0,3.0,19.0,6.0
1,595.1,46.0,211.0,22.0,13.0,51.0,26.0
2,194.8,20.0,6.0,32.0,22.0,36.0,3.0
...,...,...,...,...,...,...,...
234426,174.9,14.0,33.0,4.0,4.0,11.0,6.0
234427,174.9,14.0,33.0,4.0,4.0,11.0,6.0
234428,174.9,14.0,33.0,4.0,4.0,11.0,6.0


## Step 3: Assessment of Missingness

In [14]:
# TODO

## Step 4: Hypothesis Testing

### Set up hypotheses

**Question**: Do recipes with more ingredients (above the median) tend to have more calories than recipes with fewer ingredients?

**Null**: Recipes above and below the median ingredient count come from the same calorie distribution (any observed difference in median calories is due to random chance).

**Alternative**: Recipes with more ingredients have a different median calorie count than recipes with fewer ingredients.

**Test Statistic**: Difference in median calories (high-ingredient group − low-ingredient group)

### Summarize technical details

In [15]:
# H0: high_ing_count.calories = low_ing_count.calories
# H1: high_ing_count.calories != low_ing_count.calories
# Test statistics: diff in med calories
# Observed distribution: median calories of high_ing_count.calories
# Simulated distribution from Null distribution: N/A --> No known null distribution
# Simulated distirbution from Permutation test: median calories of high_ing_count.calories
# significance level = 0.05

### Older hypo code

In [16]:
# This is older code --> Too slow by using Pandas Groupby

# Define test statistic function
def diff_of_medians(data, groupby_col, agg_col, cat_labels):
    # Compute test statistics
    return (
        data
        .groupby(by=groupby_col)[agg_col]
        .median()
        .aggregate(lambda x: abs(x.loc[cat_labels[0]] - x.loc[cat_labels[1]]))
    )

# Compute ts
def observed_test_statistics(data):
    return diff_of_medians(
        data, 
        'high_low_ingredient_counts',
        'nutrition_calories',
        ['high', 'low']
    )

# Compute ts
def simulated_test_statistics(data):
    shuffled_df = data.copy()
    shuffled_df['high_low_ingredient_counts'] = np.random.permutation(shuffled_df['high_low_ingredient_counts'])
    return diff_of_medians(
        shuffled_df,
        'high_low_ingredient_counts',
        'nutrition_calories',
        ['high', 'low']
    )

# Compute p-value
def compute_p_value_old(data):
    # Compute observed test statistics
    observed_ts = observed_test_statistics(data)
    # Compute empirical test statistics
    simulated_ts = [simulated_test_statistics(data) for _ in range(500)]

    print(observed_ts)
    print(simulated_ts)

    # Compute p-values
    return np.mean(np.array(simulated_ts) >= observed_ts)

### Current code for hypothesis testing

In [17]:
## This is newer code --> more efficienct using NumPy vectorized ops
def hypothesis_testing(data, n_simulations=1000):
    ## Type Conversion
    # Convert series to numpy arr
    calories = data['nutrition_calories'].values

    ## Compute observed statistics
    # Pre-compute high_nutrition data as binary mask
    high_mask = (data['high_low_ingredient_counts'] == 'high').values
    
    # Use original data
    observed_ts = abs(
        # Define data with high/low nutrition using mask
        np.median(calories[high_mask]) - 
        np.median(calories[~high_mask])
    )
    
    ## Compute simulated statistics
    # Simulate using numpy only (no pandas overhead per iteration)
    n_high = high_mask.sum()

    # Permutation test in N trials
    # Generate all permutation indices at once: 
    # Generate random noises: shape (n_simulations, n_calories)
    random_noises = np.random.rand(n_simulations, len(calories))

    # Generate random permutation of indices: shape (n_simulations, n_calories)
    indices = np.argsort(random_noises, axis=1)

    # Compute median across all simulations
    simulated_ts = np.abs(
        # Define data with high/low nutrition using mask
        np.median(calories[indices[:, :n_high]], axis=1) - 
        np.median(calories[indices[:, n_high:]], axis=1)
    )
    
    ## Compute p-value
    p_val =  np.mean(simulated_ts >= observed_ts)

    ## Define sig level
    sig_level = 0.05

    ## Hypothesis testing verdict
    verdict = (
        'reject H0 and favor toward H1' 
        if p_val <= sig_level 
        else 'fail to reject H0 and inconclusive about H1'
    )

    ## Return final values
    return (observed_ts, simulated_ts, p_val, sig_level, verdict)

In [18]:
## Set up for using hypothesis function
# transform ingredient list to ingredient num and find its median
hypo_test_df = merged_df.copy()
ingred_lengths = hypo_test_df['ingredients'].apply(lambda ser: len(ser))
ingred_median = np.median(ingred_lengths)

# Create a new feature that mark a row as high
hypo_test_df['high_low_ingredient_counts'] = (
    np.where(
        ingred_lengths > ingred_median,
        'high',
        'low'
    )
)

## Perform hypothesis testing
(obs_ts, sim_ts, p_val, sig_level, conclusion) = hypothesis_testing(hypo_test_df)

display(obs_ts)
display(p_val)
display(sig_level)
display(conclusion)

# Identify Pandas groupby overhead and convert to NumPy vectorize operations for permutation test on 500 trials, reduce runtime from 1 min down to 18 secs

np.float64(113.29999999999998)

np.float64(0.0)

0.05

'reject H0 and favor toward H1'

## Step 5: Framing a Prediction Problem

Calories
Regression

Calories is a continuous numeric variable (kcal).

## Step 6: Baseline Model

In [19]:
# TODO

## Step 7: Final Model

In [20]:
# TODO

## Step 8: Fairness Analysis

In [21]:
# TODO